# Prompt 계층도

In [ ]:
"""
prompts class 계층도


BasePromptTemplate --> PipelinePromptTemplate
                       StringPromptTemplate   --> PromptTemplate
                                                  FewShotPromptTemplate
                                                  FewShotPromptWithTemplates
                       BaseChatPromptTemplate --> AutoGPTPrompt
                                                  ChatPromptTemplate --> AgentScratchPadChatPromptTemplate


  



BaseMessagePromptTemplate --> MessagesPlaceholder
                              BaseStringMessagePromptTemplate --> ChatMessagePromptTemplate
                                                                  HumanMessagePromptTemplate
                                                                  AIMessagePromptTemplate
                                                                  SystemMessagePromptTemplate
"""
None

# API Key

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

# 기본 import

In [2]:
from langchain_openai.chat_models import ChatOpenAI
from langchain_openai.llms.base import OpenAI
from langchain_core.prompts.prompt import PromptTemplate
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

In [3]:
chat = ChatOpenAI(
    temperature=0.1,
    streaming=True,
    callbacks=[
        StreamingStdOutCallbackHandler()
    ],
)

# PromptTemplate rev.

In [ ]:
# PromptTemplate을 구성하는 두가지 방법

In [5]:
# 방법1
t = PromptTemplate.from_template('What is the capital of {country}')
t.format(country='France')

'What is the capital of France'

In [7]:
# 방법2
t2 = PromptTemplate(
    template='What is the capital of {country}?',
    input_variables = ['country']  #입력변수가 무엇인지 알려줘야 함
)
t2.format(country='North Korea')

'What is the capital of North Korea?'

# FewShotPromptTemplate
모델에 예제(example) 주기

In [8]:
# 모델에게 '어떻게 대답해야 하는 지에 대한 예제(example)'를 AI 모델에게 주는 것이
# prompt 를 사용해서 '어떻게 대답해야 하는지 알려주는 것'보다 훨씬 좋다


# FewShotPromptTemplate 이 하는 일이 바로 그거다!
# - 이를 통해 예제(샘플)를 형식화(포맷) 할수 있다.
# - 이런 예제들을 데이터베이스등에 저장시켜놓고 활용할수도 있다

In [9]:
from langchain_core.prompts.few_shot import FewShotPromptTemplate

In [10]:
# 예제 없이 줬을때
chat.invoke('what do you know about France?')

France is a country located in Western Europe. It is known for its rich history, culture, and cuisine. The capital city is Paris, which is famous for landmarks such as the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral.

France is also known for its wine production, with regions such as Bordeaux, Burgundy, and Champagne producing some of the world's most renowned wines. The country is also famous for its fashion industry, with Paris being considered one of the fashion capitals of the world.

French cuisine is highly regarded internationally, with dishes such as croissants, baguettes, escargot, and coq au vin being popular around the world. The country is also known for its cheese, with varieties such as Brie, Camembert, and Roquefort being enjoyed by many.

France has a diverse landscape, ranging from the beaches of the French Riviera to the mountains of the Alps and Pyrenees. The country is also home to numerous historic sites, including the Palace of Versailles, Mont Saint-Mic

AIMessage(content="France is a country located in Western Europe. It is known for its rich history, culture, and cuisine. The capital city is Paris, which is famous for landmarks such as the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral.\n\nFrance is also known for its wine production, with regions such as Bordeaux, Burgundy, and Champagne producing some of the world's most renowned wines. The country is also famous for its fashion industry, with Paris being considered one of the fashion capitals of the world.\n\nFrench cuisine is highly regarded internationally, with dishes such as croissants, baguettes, escargot, and coq au vin being popular around the world. The country is also known for its cheese, with varieties such as Brie, Camembert, and Roquefort being enjoyed by many.\n\nFrance has a diverse landscape, ranging from the beaches of the French Riviera to the mountains of the Alps and Pyrenees. The country is also home to numerous historic sites, including the Palace of V

In [11]:
# 예제(들)
# 모델이 나에게 '이런식으로 답변해줬으면 좋겠다' 라고 제시하는 예제(example)

examples = [

    # example1
    {
        'question':'What do you know about France?',

        # 원하는 형식의 답변
        'answer':'''
          Here is what I know:
          Capital: Paris
          Language: French
          Food: Wine and Cheese
          Currency: Euro
        '''
    },  

    # example2
    {
        "question": "What do you know about Italy?",
        "answer": """
          I know this:
          Capital: Rome
          Language: Italian
          Food: Pizza and Pasta
          Currency: Euro
          """,
        },

    #example3
    {
        "question": "What do you know about Greece?",
        "answer": """
          I know this:
          Capital: Athens
          Language: Greek
          Food: Souvlaki and Feta Cheese
          Currency: Euro
          """,
    },
]

In [12]:
# FewShotTEmplate를 사용 하여 prompt 만들기

example_template = '''
    Human : {question}
    AI : {answer}
'''

# 위 {question} 과 {answer} 는 위에서 작성한 예제와 동일한 key를 사용하도록 작성

In [13]:
example_prompt = PromptTemplate.from_template(example_template)

example_prompt

PromptTemplate(input_variables=['answer', 'question'], input_types={}, partial_variables={}, template='\n    Human : {question}\n    AI : {answer}\n')

In [17]:
# print(example_prompt.format(
#     answer = examples[2]['answer'],
#     question = examples[2]['question']
# ))

print(example_prompt.format(**examples[2]))


    Human : What do you know about Greece?
    AI : 
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      



In [27]:
prompt = FewShotPromptTemplate(
    example_prompt=example_prompt, # 사용할 prompt 를 넘겨줌
    examples=examples, # 예시를 넘겨줌
    
    #suffix=
    # 포매팅된 모든 예제 마지막에 나오는 내용

    suffix = 'Human: What do you know about {country}?',
    input_variables=['country']
)

prompt

FewShotPromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, examples=[{'question': 'What do you know about France?', 'answer': '\n          Here is what I know:\n          Capital: Paris\n          Language: French\n          Food: Wine and Cheese\n          Currency: Euro\n        '}, {'question': 'What do you know about Italy?', 'answer': '\n      I know this:\n      Capital: Rome\n      Language: Italian\n      Food: Pizza and Pasta\n      Currency: Euro\n      '}, {'question': 'What do you know about Greece?', 'answer': '\n      I know this:\n      Capital: Athens\n      Language: Greek\n      Food: Souvlaki and Feta Cheese\n      Currency: Euro\n      '}], example_prompt=PromptTemplate(input_variables=['answer', 'question'], input_types={}, partial_variables={}, template='\n    Human : {question}\n    AI : {answer}\n'), suffix='Human: What do you know about {country}?')

In [20]:
print(prompt.format(country='Germany'))


    Human : What do you know about France?
    AI : 
          Here is what I know:
          Capital: Paris
          Language: French
          Food: Wine and Cheese
          Currency: Euro
        



    Human : What do you know about Italy?
    AI : 
      I know this:
      Capital: Rome
      Language: Italian
      Food: Pizza and Pasta
      Currency: Euro
      



    Human : What do you know about Greece?
    AI : 
      I know this:
      Capital: Athens
      Language: Greek
      Food: Souvlaki and Feta Cheese
      Currency: Euro
      


Human: What do you know about Germany?


In [ ]:
# step1  example 리스트를 만들고   examples
# step2  FewShotPromptTemplate 에 전달했고 examples=
# step3  어떻게 전달한 예제들을 형식화 할지 알려주었고
# step4  마지막에 질문을 포함시켰다.  suffix, input_variables


# AI 는 우리의 예제들과 똑같은 구조, 형태로 답변하게 될겁니다

In [21]:
chain = prompt | chat

In [25]:
chain.invoke({
    'country' : 'Philippine '})

AI: 
      Here is what I know:
      Capital: Manila
      Language: Filipino and English
      Food: Adobo and Sinigang
      Currency: Philippine Peso

AIMessage(content='AI: \n      Here is what I know:\n      Capital: Manila\n      Language: Filipino and English\n      Food: Adobo and Sinigang\n      Currency: Philippine Peso', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-3.5-turbo-0125', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--019be8ac-42e8-7d93-bb50-409f7de4766b', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_tokens': 38, 'total_tokens': 190, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [26]:
chain.invoke({
    'country' : 'Turkey'
})

AI: 
      I know this:
      Capital: Ankara
      Language: Turkish
      Food: Kebab and Baklava
      Currency: Turkish Lira

AIMessage(content='AI: \n      I know this:\n      Capital: Ankara\n      Language: Turkish\n      Food: Kebab and Baklava\n      Currency: Turkish Lira', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-3.5-turbo-0125', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--019be8ae-0bfa-7b12-820f-ad798b91cdda', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 152, 'output_tokens': 35, 'total_tokens': 187, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

# FewShotChatMessagePromptTemplate

In [28]:
from langchain_core.prompts.few_shot import FewShotChatMessagePromptTemplate

In [29]:
from langchain_core.prompts.chat import ChatPromptTemplate

In [34]:
examples = [

    # example1
    {
        'country':'France',

        # 원하는 형식의 답변
        'answer':'''
          Here is what I know:
          Capital: Paris
          Language: French
          Food: Wine and Cheese
          Currency: Euro
        '''
    },  

    # example2
    {
        'country':'Italy',
        "answer": """
          I know this:
          Capital: Rome
          Language: Italian
          Food: Pizza and Pasta
          Currency: Euro
          """,
        },

    #example3
    {
        'country':'Greece',
        "answer": """
          I know this:
          Capital: Athens
          Language: Greek
          Food: Souvlaki and Feta Cheese
          Currency: Euro
          """,
    },
]

In [36]:
example_prompt = ChatPromptTemplate.from_messages([ 
    ('human', 'What do you know anout {country}?'), # example에서와 동일한 key로 설정
    ('ai', '{answer}') # eample에서와 동일한 key로 설정
])

example_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

In [37]:
final_prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a geography expert, you give short answers.'),
    example_prompt,
    ('human', 'What do you know about {country}?'),
])

final_prompt.format_messages(country='Germany')

[SystemMessage(content='You are a geography expert, you give short answers.', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='What do you know anout France?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='\n          Here is what I know:\n          Capital: Paris\n          Language: French\n          Food: Wine and Cheese\n          Currency: Euro\n        ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What do you know anout Italy?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='\n          I know this:\n          Capital: Rome\n          Language: Italian\n          Food: Pizza and Pasta\n          Currency: Euro\n          ', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='What do you know anout Greece?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='\n          I know this:\n          Cap

In [39]:
chain = final_prompt | chat

In [40]:
chain.invoke({
    'country':'Brazil'
})


          I know this:
          Capital: Brasília
          Language: Portuguese
          Food: Feijoada and Brigadeiro
          Currency: Brazilian Real

AIMessage(content='\n          I know this:\n          Capital: Brasília\n          Language: Portuguese\n          Food: Feijoada and Brigadeiro\n          Currency: Brazilian Real', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-3.5-turbo-0125', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--019be8bb-c455-72c3-8649-b51dfed56f49', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 170, 'output_tokens': 32, 'total_tokens': 202, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [41]:
chain.invoke({
    'country':'South Korea'
})


          I know this:
          Capital: Seoul
          Language: Korean
          Food: Kimchi and Bibimbap
          Currency: South Korean Won

AIMessage(content='\n          I know this:\n          Capital: Seoul\n          Language: Korean\n          Food: Kimchi and Bibimbap\n          Currency: South Korean Won', additional_kwargs={}, response_metadata={'finish_reason': 'stop', 'model_name': 'gpt-3.5-turbo-0125', 'service_tier': 'default', 'model_provider': 'openai'}, id='lc_run--019be8bc-6755-7003-ac7f-2ad82d29cc77', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 171, 'output_tokens': 32, 'total_tokens': 203, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [42]:
#  때로는 수천개의 예제를 가지고 있을텐데,  이를 모두 모델에게 줄수 없는 상황이 있을수 있다.
#    이유1) 비용이 많이 든다..  많은 텍스트 땜에.
#    이유2) '허용하는 범위' 라는게 있다 => 모~든 예제들을 모델에게 줄 수는 없다.  제한이 있다 (context window)


#  그래서 예제를 선별하는 방법에 대해 배워보자


# ExampleSelector

## LengthBasedExampleSelector

In [43]:
from langchain_core.example_selectors.length_based import LengthBasedExampleSelector

In [45]:
# LengthBasedExampleSelector 는 기본적으로
# - 예제(example) 들을 형식화 할 수 있고
# - 예제의 양이 얼마나 되는지를 확인할수 있다.


# 그러면, 사용자가 설정해 놓은 세팅값에 따라 prompt 에 알맞은 예제를 골라준다.

In [47]:
print(examples)

[{'country': 'France', 'answer': '\n          Here is what I know:\n          Capital: Paris\n          Language: French\n          Food: Wine and Cheese\n          Currency: Euro\n        '}, {'country': 'Italy', 'answer': '\n          I know this:\n          Capital: Rome\n          Language: Italian\n          Food: Pizza and Pasta\n          Currency: Euro\n          '}, {'country': 'Greece', 'answer': '\n          I know this:\n          Capital: Athens\n          Language: Greek\n          Food: Souvlaki and Feta Cheese\n          Currency: Euro\n          '}]


In [48]:
example_prompt = PromptTemplate.from_template('Human: {country}\nAI:{answer}')
example_prompt

PromptTemplate(input_variables=['answer', 'country'], input_types={}, partial_variables={}, template='Human: {country}\nAI:{answer}')

In [52]:
example_selector = LengthBasedExampleSelector(
    example_prompt=example_prompt,
    examples=examples,
    max_length=180, # 예제의 양을 얼마나 허용할지 설정
                   # max_length= 값 밖의 예제는 제외 된다.
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    # examples= 대신에
    example_selector=example_selector, # max_length 로 설정한 값에 따라 예제의 양을 정해준다.

    suffix='Human: What do you know about {country}',
    input_variables=['country'],
)

print(prompt.format(country='Vietnam'))

Human: France
AI:
          Here is what I know:
          Capital: Paris
          Language: French
          Food: Wine and Cheese
          Currency: Euro
        

Human: Italy
AI:
          I know this:
          Capital: Rome
          Language: Italian
          Food: Pizza and Pasta
          Currency: Euro
          

Human: What do you know about Vietnam


In [ ]:
# ↑ 선택된 예제가 없음
# example이 formatting이 안되어있음
# 그러한 이유는 현재 max_length 값이 너무 작아서 그럼
# 여기서 단위는 token
# 즉 10token을 맥스로 잡은거임

## Custom ExampleSelector(BaseExampleSelector)

In [53]:
# 내가 원하는 대로의 example 들을 허용할수 있도록 ExampleSelector 를 제공해줄수 있다

In [54]:
from langchain_core.example_selectors.base import BaseExampleSelector

In [55]:
# BaseExampleSelector 의 구현객체를 만드려면
#  상속 받은뒤 select_examples() 과 add_example() 을 반드시 오버라이딩 해주어야 한다.

class RandomExampleSelector(BaseExampleSelector):

    def __init__(self, examples):
        self.examples = examples
        
    # select_examples()
    # 입력에 따라 어떠한 샘플을 사용할지 select 함.
    
    # 이번 예제에서는 examples 리스트 에서 random 으로 선택하게 하려 함.
    # ※ 이는 얼마든지 복잡하게 만들어 볼수도 있다.
    def select_examples(self, input_variables):
        from random import choice
        return [choice(self.examples)] #examples 에서 무작위로 선택 1개
    
    
    # add_example()
    # Add new example to store.  이미 존재하는 example 에 example 을 추가하는 method
    def add_example(self, example):
        self.examples.apped(example)





In [200]:
example_selector = RandomExampleSelector(
    examples=examples,
)

prompt = FewShotPromptTemplate(
    example_prompt=example_prompt,
    # examples= 대신에
    example_selector=example_selector, # max_length 로 설정한 값에 따라 예제의 양을 정해준다.

    suffix='Human: What do you know about {country}',
    input_variables=['country'],
)

print(prompt.format(country='Vietnam'))

Human: Italy
AI:
          I know this:
          Capital: Rome
          Language: Italian
          Food: Pizza and Pasta
          Currency: Euro
          

Human: What do you know about Vietnam


# PromptTemplate 저장/읽어오기
- load_prompt()
- prompt를 'JSON' or 'YAML' 파일로 만들 수 있다.

## JSON 파일로 저장

In [201]:
with open('prompt.json', 'w') as f:
    f.write('''
        {
        "_type":"prompt",
        "template":"What is the capital of {country}",
        "input_variables":["country"]
        }
    ''')

In [202]:
!type prompt.json


        {
        "_type":"prompt",
        "template":"What is the capital of {country}",
        "input_variables":["country"]
        }
    


In [204]:
from langchain_core.prompts.loading import load_prompt

In [205]:
prompt = load_prompt('./prompt.json')

prompt

PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='What is the capital of {country}')

In [206]:
prompt.format(country='Switzerland')

'What is the capital of Switzerland'

## YMAL 파일로 저장

In [207]:
# prompt.yaml 파일을 만든다


# name: value
#  ★ name 은 쌍따옴표 없다.  : 다음에 한칸 꼭 띄우기!   뒤에 콤마 없다
"""
_type: "prompt"
"""
None


In [208]:
with open("prompt.yaml", "w") as f:
  f.write("""
    _type: "prompt"
    template: "What is the capital of {country}"
    input_variables: ["country"]
  """)


In [210]:
prompt = load_prompt('./prompt.yaml')

prompt

PromptTemplate(input_variables=['country'], input_types={}, partial_variables={}, template='What is the capital of {country}')

In [211]:
prompt.format(country='Netherland')

'What is the capital of Netherland'

# Caching

In [212]:
# Caching 을 사용하면 모델의 응답을 저장할수 있다.
# 예를들어
# 똑같은 질문을 받는 상황이라면 그 때마다 답변생성할 필요 없이
# 이미 캐싱된 답변을 재사용 할수 있는 것이다 -->  비용절감! 

## set_llm_cache(), InMemoryCache

In [213]:
from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache 

In [214]:
set_llm_cache(InMemoryCache())  
# 세팅 이후에 llm의 모든 response가 '메모리'에 저장이 됨

In [215]:
chat = ChatOpenAI(temperature=0.1)
# cache 동작 확일은 위해, streaming x 

In [216]:
# 동일한 질문을 두번 해볼거다.  시간측정하는 함수를 준비해보자
import time
from datetime import timedelta


def check_laptime(message):
    start_time = time.time()
    response = chat.invoke(message)
    end_time = time.time()
    elapsed_time = end_time - start_time # 경과시간
    print('▶ 경과시간 %s' % (str(timedelta(seconds = elapsed_time))))
    print(f'{len(response.content)} 글자: {response.content}\n')


In [218]:
check_laptime('I want you to explain how to build pyramid')

▶ 경과시간 0:00:02.865454
1682 글자: Building a pyramid is a complex and labor-intensive process that requires careful planning and execution. Here are the general steps involved in building a pyramid:

1. Site selection: Choose a suitable location for the pyramid, taking into account factors such as the terrain, accessibility, and alignment with celestial bodies.

2. Foundation: Excavate a level area for the pyramid's base and lay down a solid foundation of stones or concrete to support the weight of the structure.

3. Construction of the core: Build a solid core of large stones or bricks in the shape of a pyramid, gradually tapering inwards as you build higher.

4. Casing stones: Add smooth, polished casing stones to the outer surface of the pyramid to give it a finished appearance and protect the core from erosion.

5. Alignment: Ensure that the pyramid is aligned with the cardinal directions and other important celestial alignments, such as the solstices or equinoxes.

6. Finishing touch

In [219]:
check_laptime('I want you to explain how to build pyramid')

▶ 경과시간 0:00:00.005060
1682 글자: Building a pyramid is a complex and labor-intensive process that requires careful planning and execution. Here are the general steps involved in building a pyramid:

1. Site selection: Choose a suitable location for the pyramid, taking into account factors such as the terrain, accessibility, and alignment with celestial bodies.

2. Foundation: Excavate a level area for the pyramid's base and lay down a solid foundation of stones or concrete to support the weight of the structure.

3. Construction of the core: Build a solid core of large stones or bricks in the shape of a pyramid, gradually tapering inwards as you build higher.

4. Casing stones: Add smooth, polished casing stones to the outer surface of the pyramid to give it a finished appearance and protect the core from erosion.

5. Alignment: Ensure that the pyramid is aligned with the cardinal directions and other important celestial alignments, such as the solstices or equinoxes.

6. Finishing touch

## set_debug()

In [220]:
from langchain_core.globals import set_debug

In [222]:
set_debug(True)

In [223]:
check_laptime('I want you to explain how to build pyramid')

[llm/start] [llm:ChatOpenAI] Entering LLM run with input:
{
  "prompts": [
    "Human: I want you to explain how to build pyramid"
  ]
}
[llm/end] [llm:ChatOpenAI] s] Exiting LLM run with output:
{
  "generations": [
    [
      {
        "text": "Building a pyramid is a complex and labor-intensive process that requires careful planning and execution. Here are the general steps involved in building a pyramid:\n\n1. Site selection: Choose a suitable location for the pyramid, taking into account factors such as the terrain, accessibility, and alignment with celestial bodies.\n\n2. Foundation: Excavate a level area for the pyramid's base and lay down a solid foundation of stones or concrete to support the weight of the structure.\n\n3. Construction of the core: Build a solid core of large stones or bricks in the shape of a pyramid, gradually tapering inwards as you build higher.\n\n4. Casing stones: Add smooth, polished casing stones to the outer surface of the pyramid to give it a finish

In [225]:
set_debug(False)

## SQLiteCache

In [226]:
from langchain_community.cache import SQLiteCache

In [228]:
set_llm_cache(SQLiteCache('cache.db'))

In [229]:
check_laptime('I want you to explain how the computer works')

▶ 경과시간 0:00:04.930633
1717 글자: A computer is a complex electronic device that processes and stores data. It works by following a series of instructions provided by the user or programmed into it. Here is a simplified explanation of how a computer works:

1. Input: The computer receives data and instructions from the user through input devices such as a keyboard, mouse, or touchscreen.

2. Processing: The central processing unit (CPU) is the brain of the computer. It processes the data and instructions received from the input devices and performs calculations and operations on them.

3. Memory: The computer has two types of memory - RAM (Random Access Memory) and storage. RAM is used to temporarily store data and instructions that the CPU needs to access quickly. Storage, such as a hard drive or solid-state drive, is used to store data and programs for long-term use.

4. Output: The computer displays the processed data and results to the user through output devices such as a monitor, pr

In [230]:
check_laptime('I want you to explain how the computer works')

▶ 경과시간 0:00:00.008100
1717 글자: A computer is a complex electronic device that processes and stores data. It works by following a series of instructions provided by the user or programmed into it. Here is a simplified explanation of how a computer works:

1. Input: The computer receives data and instructions from the user through input devices such as a keyboard, mouse, or touchscreen.

2. Processing: The central processing unit (CPU) is the brain of the computer. It processes the data and instructions received from the input devices and performs calculations and operations on them.

3. Memory: The computer has two types of memory - RAM (Random Access Memory) and storage. RAM is used to temporarily store data and instructions that the CPU needs to access quickly. Storage, such as a hard drive or solid-state drive, is used to store data and programs for long-term use.

4. Output: The computer displays the processed data and results to the user through output devices such as a monitor, pr

In [232]:
set_llm_cache(None)
set_debug(False)

# OpenAI 모델 호출 비용 확인

In [233]:
from langchain_community.callbacks.manager import get_openai_callback

In [234]:
with get_openai_callback() as usage:
    # with 블록 안에서 LLM  호출
    chat.invoke('Can you recommand me what to have for party?')
    print('🤖 USAGE : ', usage)

🤖 USAGE :  Tokens Used: 234
	Prompt Tokens: 17
		Prompt Tokens Cached: 0
	Completion Tokens: 217
		Reasoning Tokens: 0
Successful Requests: 1
Total Cost (USD): $0.000334


In [235]:
# - Prompt 가 얼마나 많은 token 을 사용했는지 (Prompt Tokens)
# - AI 결과를 위해 얼마나 많은 token 이 사용되었는지 (Completion Tokens)
# - 최종적으로 비용도 표시 (Total Cost)

In [236]:
with get_openai_callback() as usage:
    # with 블록 안에서 LLM  호출
    a = chat.invoke('Can you recommand me what to have for party?')
    b = chat.invoke('Can you recommand me what to have for dinner?')
    print('\n 📌', a.content)
    print('\n 📌', b.content)
    print('🤖 USAGE : ', usage)


 📌 Sure! Here are some popular party food and drink options that are sure to be a hit with your guests:

- Finger foods: Mini sliders, chicken wings, spring rolls, and cheese and charcuterie platters are all great options for easy-to-eat party snacks.
- Chips and dip: Serve a variety of chips with different dips such as guacamole, salsa, and spinach artichoke dip.
- Pizza: Order a selection of pizzas with different toppings to please all of your guests.
- Desserts: Cupcakes, cookies, and brownies are always a crowd-pleaser. You could also consider setting up a DIY ice cream sundae bar.
- Cocktails: Offer a selection of cocktails such as margaritas, mojitos, and cosmopolitans, as well as non-alcoholic options like mocktails and punch.

Remember to consider any dietary restrictions or preferences your guests may have when planning your party menu. Enjoy your party!

 📌 Sure! Here are a few dinner ideas:

1. Grilled chicken with roasted vegetables
2. Spaghetti with marinara sauce and gar

In [237]:
usage.total_cost

0.000479

In [238]:
usage.total_tokens

342


# Model Config 저장하고 불러오기

In [239]:
llm = OpenAI(
    temperature=0.1,
    max_tokens=450,
    model='gpt-4o'
)

llm.save('model.json')

In [240]:
!type model.json

{
    "model_name": "gpt-4o",
    "temperature": 0.1,
    "top_p": 1,
    "frequency_penalty": 0,
    "presence_penalty": 0,
    "n": 1,
    "seed": null,
    "logprobs": null,
    "max_tokens": 450,
    "_type": "openai"
}


In [241]:
from langchain_community.llms.loading import load_llm

In [243]:
chat = load_llm('model.json')

F:\KDT2508\.venv\Lib\site-packages\langchain_community\llms\openai.py:255: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain_community.chat_models import ChatOpenAI`
  warnings.warn(
F:\KDT2508\.venv\Lib\site-packages\langchain_community\llms\openai.py:1089: UserWarning: You are trying to use a chat model. This way of initializing it is no longer supported. Instead, please use: `from langchain_community.chat_models import ChatOpenAI`
  warnings.warn(


In [244]:
chat

OpenAIChat(client=APIRemovedInV1Proxy, model_kwargs={'model_name': 'gpt-4o', 'temperature': 0.1, 'top_p': 1, 'frequency_penalty': 0, 'presence_penalty': 0, 'n': 1, 'seed': None, 'logprobs': None, 'max_tokens': 450}, prefix_messages=[])